# KNN with different distance metric

In this notebook you will first find an example of how to use the Gini prametric and secondly you will find the experiments carried out to compare the performances of different distances using the KNN algorithm

In [ ]:
from sklearn.datasets import load_iris, load_wine, load_digits, load_breast_cancer, fetch_openml
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report 
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
import torch

In [ ]:
from Knn_Gini_torch import GiniDistanceTorch

## Dataset MNIST

In [ ]:
# Load MNIST
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X = mnist.data
y = mnist.target.astype(int)  # Convert to int

# Plot 3 digits
fig, axes = plt.subplots(1, 3, figsize=(5, 3))
for i, ax in enumerate(axes):
    ax.imshow(X[i].reshape(28, 28), cmap='gray')
    ax.set_title(f'Label: {y[i]}')
    ax.axis('off')
plt.show()

### How to use Sklearn's KNN with Gini prametric ?

In [ ]:
# Split data 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
print(X_train.shape, X_test.shape)

# Example usage:
model = GiniDistanceTorch(X_train, gini_param=2,  device='cuda:2')
train_distances_gini = model.compute_distances(X_train)
test_distances_gini = model.compute_distances(X_test)

# Initialize KNN classifier with k=3
knn = KNeighborsClassifier(n_neighbors=3, metric='precomputed')
knn.fit(train_distances_gini, y_train)

# Confusion matrix
from sklearn.metrics import classification_report, confusion_matrix
y_pred = knn.predict(test_distances_gini)
print(classification_report(y_test, y_pred))

del model, train_distances_gini, test_distances_gini, y_pred
torch.cuda.empty_cache()


# Standard KNN with Minkowski

In [ ]:
#Free GPU cache
import gc
gc.collect()
# Split data 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
print(X_train.shape, X_test.shape)

model_2 = GiniDistanceTorch(X_train, gini_param=2, device='cuda:2')
train_distances_minkowski = model_2.compute_minkowski_distances(X_train, p=2)
test_distances_minkowski  = model_2.compute_minkowski_distances(X_test, p=2)

knn = KNeighborsClassifier(n_neighbors=3, metric='precomputed')
knn.fit(train_distances_minkowski, y_train)

y_pred = knn.predict(test_distances_minkowski)
print(classification_report(y_test, y_pred))
del model_2, train_distances_minkowski, test_distances_minkowski, y_pred
torch.cuda.empty_cache()


# Searching for best nu

In [ ]:
def evaluate_gini_param(X_train, X_test, y_train, y_test, gini_param):
    try:
        with torch.no_grad():
            model = GiniDistanceTorch(X_train, gini_param=gini_param, device='cuda:2')
            train_distances_gini = model.compute_distances(X_train)
            test_distances_gini = model.compute_distances(X_test)

            knn = KNeighborsClassifier(n_neighbors=3, metric='precomputed')
            knn.fit(train_distances_gini, y_train)
            y_pred = knn.predict(test_distances_gini)
            f1 = f1_score(y_test, y_pred, average='weighted')
    finally:
        del model, train_distances_gini, test_distances_gini, y_pred
        gc.collect()
        try:
            torch.cuda.empty_cache()
        except RuntimeError as e:
            print("CUDA empty_cache error ignored:", e)
    return f1


In [ ]:
#Free GPU 
gc.collect()
# Split data 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
print(X_train.shape, X_test.shape)

best_gini = None
best_f1 = -1
f1_scores = []

gini_params = np.arange(1.7, 2.51, 0.1)
for gini_param in gini_params:
    print(f"Testing gini_param={gini_param:.2f}...")
    f1 = evaluate_gini_param(X_train, X_test, y_train, y_test, gini_param)
    f1_scores.append(f1)

    print(f"Gini param: {gini_param:.2f}, F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_gini = gini_param

print("\nBest Gini param:", best_gini)
print("Best F1-score:", best_f1)